<a href="https://colab.research.google.com/github/deepa22-eng/Machine-Learning-2026/blob/main/Deepa_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 3 - Deadline 2/27/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Deepa Ranabhat

**AI usage statement:**
I used Claude AI for understanding and interpretation of the code that I used to run the calculation.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Split the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

#### B)
For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

#### C)
Perform classifcation using random forest classification model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: 2
- Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:
- Accuracy
- Precision
- Receiver Operating Characteristic Area Under the Curve (ROC AUC)  

In your analysis and metrics, the high permeability class should be consider as the postive class.

Based on your analysis, which feature set gives the best results for the classifcation?

#### D) - Optional for 2 points
Repeat your analysis from C) using a $k$ nearest neighbors classifer (i.e., $k$ nearest neighbors vote). Use the default value of 5 neighbors.

Do you obtain the same results as in C)?



In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dataset_1 = pd.read_csv("model_data.csv")
dataset_2 = pd.read_csv("2d_features.csv")

In [ ]:
dataset_1


In [ ]:
dataset_2

In [ ]:
dataset_1['P_app'] = dataset_2['P_app']
dataset_1['Smiles'] = dataset_2['Smiles']

In [ ]:
dataset_1

A) Splitting the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

In [ ]:
Permeable_cutoff = (7)
Low_label = 0
High_label = +1
Permeable_key_str = f'Permeability High({High_label:})/Low({Low_label:})'
dataset_1[Permeable_key_str] = [High_label if p > Permeable_cutoff else Low_label for p in dataset_1['P_app']]

Number_Permeable_High = np.sum(dataset_1[Permeable_key_str] == +1)
Number_Permeable_Low = np.sum(dataset_1[Permeable_key_str] == 0)

print("Key:",Permeable_key_str)

print("Number with high permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_High))
print("Number with low permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_Low))

print("")

dataset_1[['P_app', Permeable_key_str] ]

B) For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

In [ ]:
# the %%capture command will surpress output to screen
%%capture
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit mols2grid

In [ ]:
import mols2grid


In [ ]:
dataset_1['P_app(nm/s)'] = dataset_1['P_app'].round(2)

In [ ]:
# by passing the dataframe, and giving the column with the SMILES string, we can
# plot all molecules
# we can also add information to the figures by using the subset variable
mols2grid.display(dataset_1,smiles_col='Smiles',subset=["img", 'P_app(nm/s)' ,Permeable_key_str ])

C) Perform classifcation using random forest classification model using the following hyperparameters:

Number of tree: 100
Maximum depth of each tree: 2
Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)
You should perform the classification using 3 different feature set:

2D RDKit features
3D Ensemble features
Combined set of 2D and 3D features
For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:

Accuracy
Precision
Receiver Operating Characteristic Area Under the Curve (ROC AUC)
In your analysis and metrics, the high permeability class should be consider as the postive class.



In [ ]:
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, ShuffleSplit
from sklearn import metrics

In [ ]:
features_2d = [
    'Molecular Weight (MW)',
    'CharVol (characteristic volume)',
    'Flexibility (number of rotatable bonds / number of bonds)',
    'Number of Heavy Atoms (HA)',
    'RingAtoms',
    'Halogens',
    'HeteroAtoms',
    'RotBonds (NRotB)',
    'AllBonds',
    'RingCount',
    'NumStereo',
    'Fraction of sp3 Carbon Atoms (FSP3)',
    'Hydrogen Bond Donors (HBD)',
    'Hydrogen Bond Acceptors (HBA)',
    'cLogD^7.4',
    'Topological polar surface area (TPSA)',
    'Total non-polar surface area (TNSA)'
]

features_3d = [
    'Ensemble_Average_PSA_Chloroform_ANI',
    'Ensemble_Average_Num_IMHB_Chloroform_ANI',
    'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]

features_combined = features_2d + features_3d

For 2D-Features

In [ ]:
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, ShuffleSplit
from sklearn import metrics

# Features and target
features = dataset_1[features_2d].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_2d) * 0.5)
print(f"Number of 2D features: {len(features_2d)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== 2D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 3D-features

In [ ]:
# Features and target
features = dataset_1[features_3d].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_3d) * 0.5)
print(f"Number of 3D features: {len(features_3d)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== 3D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 2D and 3D combined features

In [ ]:
# Features and target
features = dataset_1[features_combined].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_combined) * 0.5)
print(f"Number of combined features: {len(features_combined)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== Combined RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

The 3D ensembles features gives the best calculation result among three. The 3D ensemble features (PSA, IMHB, radius of gyration)
obtained from MD simulations are the most informative for
predicting passive permeability. This makes physical sense
because permeability depends on the 3D conformation and
dynamic behavior of the molecule in a membrane-like
environment, which the 2D RDKit descriptors cannot capture.

Interestingly, adding 2D features to the 3D features
(combined) actually slightly decreased performance compared
to 3D alone, suggesting the 2D features may add noise
rather than useful information for this small dataset of
32 compounds.

D) Optional
 Analysing a  𝑘  nearest neighbors classifer (i.e.,  𝑘  nearest neighbors vote) by using the default value of 5 neighbors.


For 2D- features


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Features and target
features = dataset_1[features_2d].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - 2D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 3D- features

In [ ]:


# Features and target
features = dataset_1[features_3d].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - 3D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 2D and 3D combined features.

In [ ]:


# Features and target
features = dataset_1[features_combined].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - Combined RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

In KNN , I cant get same result as in random forest. However,  3D- ensembles gives the best reult among three like as in random forest analyis.
If we compare both model the random forest ROU value is higher than KNN model. So for 32- compound data set random forest model seem work well.